<a href="https://colab.research.google.com/github/Noors-lab/Model-s_summaries/blob/main/training__multi_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ─── Cell 1: Phase 0 — Freeze the split (by clip, not window) ───
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import glob
import random
import json

random.seed(42)  # reproducible split

BASE_DIR      = "/content/drive/MyDrive/tame/dataset"
NORMAL_DIR    = f"{BASE_DIR}/normal"
SHOPLIFT_DIR  = f"{BASE_DIR}/shoplifting"
SPLIT_OUT     = f"{BASE_DIR}/split.json"   # saved so V1/V2/etc reuse the SAME split

def list_clips(folder):
    return sorted(glob.glob(f"{folder}/*.npy"))

normal_clips   = list_clips(NORMAL_DIR)
shoplift_clips = list_clips(SHOPLIFT_DIR)

print(f"Normal clips: {len(normal_clips)}")
print(f"Shoplifting clips: {len(shoplift_clips)}")

def split_clips(clips, train_r=0.70, val_r=0.15):
    clips = clips.copy()
    random.shuffle(clips)
    n = len(clips)
    n_train = int(n * train_r)
    n_val   = int(n * val_r)
    return {
        "train": clips[:n_train],
        "val":   clips[n_train:n_train + n_val],
        "test":  clips[n_train + n_val:]
    }

normal_split   = split_clips(normal_clips)
shoplift_split = split_clips(shoplift_clips)

split = {
    "train": [{"path": p, "label": 0} for p in normal_split["train"]] +
             [{"path": p, "label": 1} for p in shoplift_split["train"]],
    "val":   [{"path": p, "label": 0} for p in normal_split["val"]] +
             [{"path": p, "label": 1} for p in shoplift_split["val"]],
    "test":  [{"path": p, "label": 0} for p in normal_split["test"]] +
             [{"path": p, "label": 1} for p in shoplift_split["test"]],
}

for k in split:
    random.shuffle(split[k])

# Save split to Drive so V1, V2, ... V5 all train/test on the EXACT same clips
with open(SPLIT_OUT, "w") as f:
    json.dump(split, f, indent=2)

print("\n─── Split summary ───")
for k, v in split.items():
    n_pos = sum(1 for x in v if x["label"] == 1)
    n_neg = sum(1 for x in v if x["label"] == 0)
    print(f"{k:5s}: {len(v):4d} clips  (shoplifting={n_pos}, normal={n_neg})")

print(f"\nSplit frozen and saved to: {SPLIT_OUT}")
print("Do not regenerate this file for later experiments — always load it.")

Mounted at /content/drive
Normal clips: 874
Shoplifting clips: 874

─── Split summary ───
train: 1222 clips  (shoplifting=611, normal=611)
val  :  262 clips  (shoplifting=131, normal=131)
test :  264 clips  (shoplifting=132, normal=132)

Split frozen and saved to: /content/drive/MyDrive/tame/dataset/split.json
Do not regenerate this file for later experiments — always load it.


# the baseline model V0

In [ ]:
# ─── Cell 2: V0 — Baseline training (no windowing, no augmentation) ───
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

BASE_DIR  = "/content/drive/MyDrive/tame/dataset"
SPLIT_PATH = f"{BASE_DIR}/split.json"
SEQ_LEN    = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

with open(SPLIT_PATH) as f:
    split = json.load(f)

# ─── Dataset ───
class ClipDataset(Dataset):
    def __init__(self, entries, seq_len=50):
        self.entries = entries
        self.seq_len = seq_len

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        seq = np.load(entry["path"]).astype(np.float32)  # [T, 34]

        if seq.shape[0] >= self.seq_len:
            seq = seq[-self.seq_len:]  # take last N frames (closest to event)
        else:
            pad = np.zeros((self.seq_len - seq.shape[0], seq.shape[1]), dtype=np.float32)
            seq = np.vstack([seq, pad])

        return torch.tensor(seq, dtype=torch.float32), torch.tensor(entry["label"], dtype=torch.float32)

train_ds = ClipDataset(split["train"], SEQ_LEN)
val_ds   = ClipDataset(split["val"], SEQ_LEN)
test_ds  = ClipDataset(split["test"], SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

# ─── Compute normalization stats from TRAIN split only (no leakage) ───
print("Computing normalization stats from train split...")
all_train_seqs = []
for entry in split["train"]:
    seq = np.load(entry["path"]).astype(np.float32)
    all_train_seqs.append(seq)
all_train_concat = np.vstack(all_train_seqs)
X_mean = all_train_concat.mean(axis=0)
X_std  = all_train_concat.std(axis=0)
np.save(f"{BASE_DIR}/X_mean_v0.npy", X_mean)
np.save(f"{BASE_DIR}/X_std_v0.npy", X_std)
print("Saved X_mean_v0.npy / X_std_v0.npy")

X_mean_t = torch.tensor(X_mean, dtype=torch.float32).to(device)
X_std_t  = torch.tensor(X_std, dtype=torch.float32).to(device)

# ─── Model (same architecture as server.py) ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :]).squeeze()

model = ShopliftingLSTM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

# ─── Training loop with early stopping on val loss ───
EPOCHS = 50
PATIENCE = 8
best_val_loss = float("inf")
patience_counter = 0
best_model_path = f"{BASE_DIR}/vigiq_v0_best.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            x = (x - X_mean_t) / (X_std_t + 1e-8)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
    val_loss /= len(val_ds)

    print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest val_loss: {best_val_loss:.4f} — model saved to {best_model_path}")

# ─── Evaluation on TEST set (frozen, untouched until now) ───
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(y.numpy().astype(int).tolist())

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average=None, labels=[0, 1])
cm = confusion_matrix(all_labels, all_preds)

print("\n─── V0 BASELINE — TEST SET RESULTS ───")
print(f"Accuracy: {acc:.4f}\n")
print("Class 0 = Normal, Class 1 = Shoplifting")
print(f"Normal      — precision={precision[0]:.3f}  recall={recall[0]:.3f}  f1={f1[0]:.3f}")
print(f"Shoplifting — precision={precision[1]:.3f}  recall={recall[1]:.3f}  f1={f1[1]:.3f}")
print(f"\nConfusion matrix:\n{cm}")
print(f"           (rows=actual, cols=predicted, order=[normal, shoplifting])")

print("\n" + classification_report(all_labels, all_preds, target_names=["Normal", "Shoplifting"]))

Using device: cuda
Computing normalization stats from train split...
Saved X_mean_v0.npy / X_std_v0.npy
Epoch  1 | train_loss=0.6867 | val_loss=0.6566
Epoch  2 | train_loss=0.5675 | val_loss=0.5398
Epoch  3 | train_loss=0.4890 | val_loss=0.5186
Epoch  4 | train_loss=0.4823 | val_loss=0.5399
Epoch  5 | train_loss=0.4780 | val_loss=0.5110
Epoch  6 | train_loss=0.4802 | val_loss=0.5453
Epoch  7 | train_loss=0.4703 | val_loss=0.5311
Epoch  8 | train_loss=0.4675 | val_loss=0.5472
Epoch  9 | train_loss=0.4678 | val_loss=0.5310
Epoch 10 | train_loss=0.4680 | val_loss=0.5281
Epoch 11 | train_loss=0.4688 | val_loss=0.5299
Epoch 12 | train_loss=0.4676 | val_loss=0.5415
Epoch 13 | train_loss=0.4733 | val_loss=0.5190
Early stopping at epoch 13

Best val_loss: 0.5110 — model saved to /content/drive/MyDrive/tame/dataset/vigiq_v0_best.pth

─── V0 BASELINE — TEST SET RESULTS ───
Accuracy: 0.8144

Class 0 = Normal, Class 1 = Shoplifting
Normal      — precision=1.000  recall=0.629  f1=0.772
Shoplifting 

# sliding window V1

In [ ]:
# ─── Cell 3: V1 — Sliding windows (clip-level label inherited, fast version) ───
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

BASE_DIR   = "/content/drive/MyDrive/tame/dataset"
SPLIT_PATH = f"{BASE_DIR}/split.json"
WINDOW_LEN = 50
STRIDE     = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

with open(SPLIT_PATH) as f:
    split = json.load(f)

# ─── Generate overlapping windows from each clip ───
def make_windows(entries, window_len=50, stride=10, is_train=False):
    windowed = []
    for entry in entries:
        seq = np.load(entry["path"]).astype(np.float32)  # [T, 34]
        T = seq.shape[0]

        if T <= window_len:
            # too short to window — pad once, single window
            pad = np.zeros((window_len - T, seq.shape[1]), dtype=np.float32)
            win = np.vstack([seq, pad])
            windowed.append({"seq": win, "label": entry["label"]})
            continue

        for start in range(0, T - window_len + 1, stride):
            win = seq[start:start + window_len]
            windowed.append({"seq": win, "label": entry["label"]})

        # ensure last window (tail of clip) is always included
        if (T - window_len) % stride != 0:
            win = seq[-window_len:]
            windowed.append({"seq": win, "label": entry["label"]})

    return windowed

print("Generating sliding windows...")
train_windows = make_windows(split["train"], WINDOW_LEN, STRIDE, is_train=True)
val_windows   = make_windows(split["val"], WINDOW_LEN, STRIDE)
test_windows  = make_windows(split["test"], WINDOW_LEN, STRIDE)

print(f"Train: {len(split['train'])} clips -> {len(train_windows)} windows")
print(f"Val:   {len(split['val'])} clips -> {len(val_windows)} windows")
print(f"Test:  {len(split['test'])} clips -> {len(test_windows)} windows")

# ─── Dataset ───
class WindowDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows
    def __len__(self):
        return len(self.windows)
    def __getitem__(self, idx):
        w = self.windows[idx]
        return torch.tensor(w["seq"], dtype=torch.float32), torch.tensor(w["label"], dtype=torch.float32)

train_ds = WindowDataset(train_windows)
val_ds   = WindowDataset(val_windows)
test_ds  = WindowDataset(test_windows)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

# ─── Normalization stats from TRAIN windows only ───
print("Computing normalization stats from train windows...")
all_train_seqs = np.vstack([w["seq"] for w in train_windows])
X_mean = all_train_seqs.mean(axis=0)
X_std  = all_train_seqs.std(axis=0)
np.save(f"{BASE_DIR}/X_mean_v1.npy", X_mean)
np.save(f"{BASE_DIR}/X_std_v1.npy", X_std)

X_mean_t = torch.tensor(X_mean, dtype=torch.float32).to(device)
X_std_t  = torch.tensor(X_std, dtype=torch.float32).to(device)

# ─── Model (identical architecture) ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :]).squeeze()

model = ShopliftingLSTM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

# ─── Training loop ───
EPOCHS = 30  # windows = way more data per epoch, fewer epochs needed
PATIENCE = 5
best_val_loss = float("inf")
patience_counter = 0
best_model_path = f"{BASE_DIR}/vigiq_v1_best.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            x = (x - X_mean_t) / (X_std_t + 1e-8)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
    val_loss /= len(val_ds)

    print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest val_loss: {best_val_loss:.4f} — model saved to {best_model_path}")

# ─── Evaluation on TEST windows ───
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(y.numpy().astype(int).tolist())

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average=None, labels=[0, 1])
cm = confusion_matrix(all_labels, all_preds)

print("\n─── V1 SLIDING WINDOWS — TEST SET RESULTS (window-level) ───")
print(f"Accuracy: {acc:.4f}\n")
print("Class 0 = Normal, Class 1 = Shoplifting")
print(f"Normal      — precision={precision[0]:.3f}  recall={recall[0]:.3f}  f1={f1[0]:.3f}")
print(f"Shoplifting — precision={precision[1]:.3f}  recall={recall[1]:.3f}  f1={f1[1]:.3f}")
print(f"\nConfusion matrix:\n{cm}")
print(f"           (rows=actual, cols=predicted, order=[normal, shoplifting])")

print("\n" + classification_report(all_labels, all_preds, target_names=["Normal", "Shoplifting"]))

Using device: cuda
Generating sliding windows...
Train: 1222 clips -> 1222 windows
Val:   262 clips -> 262 windows
Test:  264 clips -> 264 windows
Computing normalization stats from train windows...
Epoch  1 | train_loss=0.6868 | val_loss=0.6635
Epoch  2 | train_loss=0.5772 | val_loss=0.5411
Epoch  3 | train_loss=0.4962 | val_loss=0.5114
Epoch  4 | train_loss=0.4777 | val_loss=0.5231
Epoch  5 | train_loss=0.4735 | val_loss=0.5304
Epoch  6 | train_loss=0.4699 | val_loss=0.5228
Epoch  7 | train_loss=0.4739 | val_loss=0.5603
Epoch  8 | train_loss=0.4736 | val_loss=0.5143
Early stopping at epoch 8

Best val_loss: 0.5114 — model saved to /content/drive/MyDrive/tame/dataset/vigiq_v1_best.pth

─── V1 SLIDING WINDOWS — TEST SET RESULTS (window-level) ───
Accuracy: 0.8144

Class 0 = Normal, Class 1 = Shoplifting
Normal      — precision=0.977  recall=0.644  f1=0.776
Shoplifting — precision=0.734  recall=0.985  f1=0.841

Confusion matrix:
[[ 85  47]
 [  2 130]]
           (rows=actual, cols=predi

# V2 adding velocity

In [ ]:
# ─── Cell 4a: V2a — Add velocity only (position + velocity, no acceleration) ───
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

BASE_DIR   = "/content/drive/MyDrive/tame/dataset"
SPLIT_PATH = f"{BASE_DIR}/split.json"
SEQ_LEN    = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

with open(SPLIT_PATH) as f:
    split = json.load(f)

# ─── Feature engineering: position + velocity only ───
def add_motion_features(seq):
    """
    seq: [T, 34] normalized (x,y) positions, 17 keypoints
    returns: [T, 68] -> position(34) + velocity(34)
    """
    velocity = np.zeros_like(seq)
    velocity[1:] = seq[1:] - seq[:-1]  # velocity[0] stays 0 (no prior frame)
    return np.concatenate([seq, velocity], axis=1)  # [T, 68]

# ─── Dataset ───
class ClipDatasetMotion(Dataset):
    def __init__(self, entries, seq_len=50):
        self.entries = entries
        self.seq_len = seq_len

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        seq = np.load(entry["path"]).astype(np.float32)  # [T, 34]

        if seq.shape[0] >= self.seq_len:
            seq = seq[-self.seq_len:]
        else:
            pad = np.zeros((self.seq_len - seq.shape[0], seq.shape[1]), dtype=np.float32)
            seq = np.vstack([seq, pad])

        # compute velocity AFTER padding so padded frames -> zero velocity too
        seq_with_motion = add_motion_features(seq)  # [50, 68]

        return torch.tensor(seq_with_motion, dtype=torch.float32), torch.tensor(entry["label"], dtype=torch.float32)

train_ds = ClipDatasetMotion(split["train"], SEQ_LEN)
val_ds   = ClipDatasetMotion(split["val"], SEQ_LEN)
test_ds  = ClipDatasetMotion(split["test"], SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

# ─── Normalization stats from TRAIN split (now over 68 features) ───
print("Computing normalization stats from train split (position+velocity)...")
all_train_seqs = []
for entry in split["train"]:
    seq = np.load(entry["path"]).astype(np.float32)
    if seq.shape[0] >= SEQ_LEN:
        seq = seq[-SEQ_LEN:]
    else:
        pad = np.zeros((SEQ_LEN - seq.shape[0], seq.shape[1]), dtype=np.float32)
        seq = np.vstack([seq, pad])
    seq_motion = add_motion_features(seq)
    all_train_seqs.append(seq_motion)
all_train_concat = np.vstack(all_train_seqs)
X_mean = all_train_concat.mean(axis=0)
X_std  = all_train_concat.std(axis=0)
np.save(f"{BASE_DIR}/X_mean_v2a.npy", X_mean)
np.save(f"{BASE_DIR}/X_std_v2a.npy", X_std)
print(f"Feature dim: {X_mean.shape[0]} (should be 68)")

X_mean_t = torch.tensor(X_mean, dtype=torch.float32).to(device)
X_std_t  = torch.tensor(X_std, dtype=torch.float32).to(device)

# ─── Model — same architecture, input_size now 68 instead of 34 ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=68, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :]).squeeze()

model = ShopliftingLSTM(input_size=68).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

# ─── Training loop ───
EPOCHS = 50
PATIENCE = 8
best_val_loss = float("inf")
patience_counter = 0
best_model_path = f"{BASE_DIR}/vigiq_v2a_best.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            x = (x - X_mean_t) / (X_std_t + 1e-8)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
    val_loss /= len(val_ds)

    print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest val_loss: {best_val_loss:.4f} — model saved to {best_model_path}")

# ─── Evaluation on TEST set ───
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(y.numpy().astype(int).tolist())

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average=None, labels=[0, 1])
cm = confusion_matrix(all_labels, all_preds)

print("\n─── V2a POSITION+VELOCITY — TEST SET RESULTS ───")
print(f"Accuracy: {acc:.4f}\n")
print("Class 0 = Normal, Class 1 = Shoplifting")
print(f"Normal      — precision={precision[0]:.3f}  recall={recall[0]:.3f}  f1={f1[0]:.3f}")
print(f"Shoplifting — precision={precision[1]:.3f}  recall={recall[1]:.3f}  f1={f1[1]:.3f}")
print(f"\nConfusion matrix:\n{cm}")
print(f"           (rows=actual, cols=predicted, order=[normal, shoplifting])")

print("\n" + classification_report(all_labels, all_preds, target_names=["Normal", "Shoplifting"]))

Using device: cuda
Computing normalization stats from train split (position+velocity)...
Feature dim: 68 (should be 68)
Epoch  1 | train_loss=0.6880 | val_loss=0.6662
Epoch  2 | train_loss=0.5800 | val_loss=0.5438
Epoch  3 | train_loss=0.4847 | val_loss=0.5164
Epoch  4 | train_loss=0.4763 | val_loss=0.5168
Epoch  5 | train_loss=0.4760 | val_loss=0.5092
Epoch  6 | train_loss=0.4733 | val_loss=0.5055
Epoch  7 | train_loss=0.4678 | val_loss=0.5219
Epoch  8 | train_loss=0.4704 | val_loss=0.5157
Epoch  9 | train_loss=0.4647 | val_loss=0.5261
Epoch 10 | train_loss=0.4695 | val_loss=0.5036
Epoch 11 | train_loss=0.4601 | val_loss=0.5259
Epoch 12 | train_loss=0.4618 | val_loss=0.5255
Epoch 13 | train_loss=0.4639 | val_loss=0.5126
Epoch 14 | train_loss=0.4628 | val_loss=0.5218
Epoch 15 | train_loss=0.4571 | val_loss=0.5188
Epoch 16 | train_loss=0.4547 | val_loss=0.5324
Epoch 17 | train_loss=0.4603 | val_loss=0.4997
Epoch 18 | train_loss=0.4601 | val_loss=0.5125
Epoch 19 | train_loss=0.4516 | val

# v3 velocity + acceleration

In [ ]:
# ─── Cell 4: V2 — Add velocity + acceleration features ───
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

BASE_DIR   = "/content/drive/MyDrive/tame/dataset"
SPLIT_PATH = f"{BASE_DIR}/split.json"
SEQ_LEN    = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

with open(SPLIT_PATH) as f:
    split = json.load(f)

# ─── Feature engineering: position + velocity + acceleration ───
def add_motion_features(seq):
    """
    seq: [T, 34] normalized (x,y) positions, 17 keypoints
    returns: [T, 102] -> position(34) + velocity(34) + acceleration(34)
    """
    T, D = seq.shape

    velocity = np.zeros_like(seq)
    velocity[1:] = seq[1:] - seq[:-1]  # velocity[0] stays 0 (no prior frame)

    acceleration = np.zeros_like(seq)
    acceleration[1:] = velocity[1:] - velocity[:-1]  # accel[0] stays 0

    return np.concatenate([seq, velocity, acceleration], axis=1)  # [T, 102]

# ─── Dataset ───
class ClipDatasetMotion(Dataset):
    def __init__(self, entries, seq_len=50):
        self.entries = entries
        self.seq_len = seq_len

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        seq = np.load(entry["path"]).astype(np.float32)  # [T, 34]

        if seq.shape[0] >= self.seq_len:
            seq = seq[-self.seq_len:]
        else:
            pad = np.zeros((self.seq_len - seq.shape[0], seq.shape[1]), dtype=np.float32)
            seq = np.vstack([seq, pad])

        # NOTE: compute motion AFTER padding, so padded (zero) frames produce
        # zero velocity/acceleration too (rather than a fake "jump to zero")
        seq_with_motion = add_motion_features(seq)  # [50, 102]

        return torch.tensor(seq_with_motion, dtype=torch.float32), torch.tensor(entry["label"], dtype=torch.float32)

train_ds = ClipDatasetMotion(split["train"], SEQ_LEN)
val_ds   = ClipDatasetMotion(split["val"], SEQ_LEN)
test_ds  = ClipDatasetMotion(split["test"], SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

# ─── Normalization stats from TRAIN split (now over 102 features) ───
print("Computing normalization stats from train split (position+velocity+accel)...")
all_train_seqs = []
for entry in split["train"]:
    seq = np.load(entry["path"]).astype(np.float32)
    if seq.shape[0] >= SEQ_LEN:
        seq = seq[-SEQ_LEN:]
    else:
        pad = np.zeros((SEQ_LEN - seq.shape[0], seq.shape[1]), dtype=np.float32)
        seq = np.vstack([seq, pad])
    seq_motion = add_motion_features(seq)
    all_train_seqs.append(seq_motion)
all_train_concat = np.vstack(all_train_seqs)
X_mean = all_train_concat.mean(axis=0)
X_std  = all_train_concat.std(axis=0)
np.save(f"{BASE_DIR}/X_mean_v2.npy", X_mean)
np.save(f"{BASE_DIR}/X_std_v2.npy", X_std)
print(f"Feature dim: {X_mean.shape[0]} (should be 102)")

X_mean_t = torch.tensor(X_mean, dtype=torch.float32).to(device)
X_std_t  = torch.tensor(X_std, dtype=torch.float32).to(device)

# ─── Model — same architecture, input_size now 102 instead of 34 ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=102, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :]).squeeze()

model = ShopliftingLSTM(input_size=102).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

# ─── Training loop ───
EPOCHS = 50
PATIENCE = 8
best_val_loss = float("inf")
patience_counter = 0
best_model_path = f"{BASE_DIR}/vigiq_v2_best.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            x = (x - X_mean_t) / (X_std_t + 1e-8)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
    val_loss /= len(val_ds)

    print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest val_loss: {best_val_loss:.4f} — model saved to {best_model_path}")

# ─── Evaluation on TEST set ───
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(y.numpy().astype(int).tolist())

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average=None, labels=[0, 1])
cm = confusion_matrix(all_labels, all_preds)

print("\n─── V2 POSITION+VELOCITY+ACCEL — TEST SET RESULTS ───")
print(f"Accuracy: {acc:.4f}\n")
print("Class 0 = Normal, Class 1 = Shoplifting")
print(f"Normal      — precision={precision[0]:.3f}  recall={recall[0]:.3f}  f1={f1[0]:.3f}")
print(f"Shoplifting — precision={precision[1]:.3f}  recall={recall[1]:.3f}  f1={f1[1]:.3f}")
print(f"\nConfusion matrix:\n{cm}")
print(f"           (rows=actual, cols=predicted, order=[normal, shoplifting])")

print("\n" + classification_report(all_labels, all_preds, target_names=["Normal", "Shoplifting"]))

Using device: cuda
Computing normalization stats from train split (position+velocity+accel)...
Feature dim: 102 (should be 102)
Epoch  1 | train_loss=0.6850 | val_loss=0.6555
Epoch  2 | train_loss=0.5618 | val_loss=0.5142
Epoch  3 | train_loss=0.4886 | val_loss=0.5467
Epoch  4 | train_loss=0.4808 | val_loss=0.5212
Epoch  5 | train_loss=0.4687 | val_loss=0.5284
Epoch  6 | train_loss=0.4744 | val_loss=0.5111
Epoch  7 | train_loss=0.4714 | val_loss=0.5180
Epoch  8 | train_loss=0.4687 | val_loss=0.5390
Epoch  9 | train_loss=0.4700 | val_loss=0.5106
Epoch 10 | train_loss=0.4673 | val_loss=0.5182
Epoch 11 | train_loss=0.4632 | val_loss=0.5402
Epoch 12 | train_loss=0.4742 | val_loss=0.5026
Epoch 13 | train_loss=0.4667 | val_loss=0.5206
Epoch 14 | train_loss=0.4669 | val_loss=0.5233
Epoch 15 | train_loss=0.4643 | val_loss=0.5341
Epoch 16 | train_loss=0.4650 | val_loss=0.5091
Epoch 17 | train_loss=0.4667 | val_loss=0.5147
Epoch 18 | train_loss=0.4648 | val_loss=0.5159
Epoch 19 | train_loss=0.45

# false positive investigation

In [ ]:
# ─── Cell 5: Option C — Inspect false positives (normal clips misclassified as shoplifting) ───
import json
import numpy as np
import torch
import torch.nn as nn

BASE_DIR   = "/content/drive/MyDrive/tame/dataset"
SPLIT_PATH = f"{BASE_DIR}/split.json"
SEQ_LEN    = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(SPLIT_PATH) as f:
    split = json.load(f)

# ─── Load V0 model (position-only, since that's our cleanest baseline) ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :]).squeeze()

model = ShopliftingLSTM().to(device)
model.load_state_dict(torch.load(f"{BASE_DIR}/vigiq_v0_best.pth", map_location=device))
model.eval()

X_mean = np.load(f"{BASE_DIR}/X_mean_v0.npy")
X_std  = np.load(f"{BASE_DIR}/X_std_v0.npy")
X_mean_t = torch.tensor(X_mean, dtype=torch.float32).to(device)
X_std_t  = torch.tensor(X_std, dtype=torch.float32).to(device)

def load_clip(path):
    seq = np.load(path).astype(np.float32)
    if seq.shape[0] >= SEQ_LEN:
        seq = seq[-SEQ_LEN:]
    else:
        pad = np.zeros((SEQ_LEN - seq.shape[0], seq.shape[1]), dtype=np.float32)
        seq = np.vstack([seq, pad])
    return seq

# ─── Run inference on test set, collect false positives + false negatives ───
false_positives = []  # normal clips predicted as shoplifting
false_negatives = []  # shoplifting clips predicted as normal

with torch.no_grad():
    for entry in split["test"]:
        seq = load_clip(entry["path"])
        x = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        logit = model(x)
        prob = torch.sigmoid(logit).item()
        pred = 1 if prob >= 0.5 else 0

        if entry["label"] == 0 and pred == 1:
            false_positives.append({"path": entry["path"], "confidence": prob})
        elif entry["label"] == 1 and pred == 0:
            false_negatives.append({"path": entry["path"], "confidence": prob})

false_positives.sort(key=lambda x: -x["confidence"])  # most confidently wrong first
false_negatives.sort(key=lambda x: x["confidence"])

print(f"False positives (normal -> predicted shoplifting): {len(false_positives)}")
print(f"False negatives (shoplifting -> predicted normal): {len(false_negatives)}")

print("\n─── Top 15 most confident FALSE POSITIVES (normal clips model is SURE are shoplifting) ───")
for fp in false_positives[:15]:
    filename = fp["path"].split("/")[-1]
    print(f"  {filename}  (model confidence: {fp['confidence']:.3f})")

print("\n─── FALSE NEGATIVES (shoplifting clips model missed) ───")
for fn in false_negatives[:15]:
    filename = fn["path"].split("/")[-1]
    print(f"  {filename}  (model confidence: {fn['confidence']:.3f})")

# Save full lists to Drive for reference
with open(f"{BASE_DIR}/false_positives.json", "w") as f:
    json.dump(false_positives, f, indent=2)
with open(f"{BASE_DIR}/false_negatives.json", "w") as f:
    json.dump(false_negatives, f, indent=2)
print(f"\nFull lists saved to false_positives.json / false_negatives.json")

False positives (normal -> predicted shoplifting): 49
False negatives (shoplifting -> predicted normal): 0

─── Top 15 most confident FALSE POSITIVES (normal clips model is SURE are shoplifting) ───
  normal_600.npy  (model confidence: 0.685)
  normal_125.npy  (model confidence: 0.685)
  normal_429.npy  (model confidence: 0.683)
  normal_434.npy  (model confidence: 0.682)
  normal_771.npy  (model confidence: 0.681)
  normal_345.npy  (model confidence: 0.681)
  normal_463.npy  (model confidence: 0.681)
  normal_745.npy  (model confidence: 0.681)
  normal_200.npy  (model confidence: 0.681)
  normal_716.npy  (model confidence: 0.680)
  normal_408.npy  (model confidence: 0.680)
  normal_854.npy  (model confidence: 0.680)
  normal_188.npy  (model confidence: 0.679)
  normal_799.npy  (model confidence: 0.679)
  normal_691.npy  (model confidence: 0.679)

─── FALSE NEGATIVES (shoplifting clips model missed) ───

Full lists saved to false_positives.json / false_negatives.json


# threshold investigation

In [ ]:
# ─── Cell 6: Threshold sweep on V0 predictions (no retraining needed) ───
import json
import numpy as np
import torch
import torch.nn as nn

BASE_DIR   = "/content/drive/MyDrive/tame/dataset"
SPLIT_PATH = f"{BASE_DIR}/split.json"
SEQ_LEN    = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(SPLIT_PATH) as f:
    split = json.load(f)

class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :]).squeeze()

model = ShopliftingLSTM().to(device)
model.load_state_dict(torch.load(f"{BASE_DIR}/vigiq_v0_best.pth", map_location=device))
model.eval()

X_mean = np.load(f"{BASE_DIR}/X_mean_v0.npy")
X_std  = np.load(f"{BASE_DIR}/X_std_v0.npy")
X_mean_t = torch.tensor(X_mean, dtype=torch.float32).to(device)
X_std_t  = torch.tensor(X_std, dtype=torch.float32).to(device)

def load_clip(path):
    seq = np.load(path).astype(np.float32)
    if seq.shape[0] >= SEQ_LEN:
        seq = seq[-SEQ_LEN:]
    else:
        pad = np.zeros((SEQ_LEN - seq.shape[0], seq.shape[1]), dtype=np.float32)
        seq = np.vstack([seq, pad])
    return seq

all_probs, all_labels = [], []
with torch.no_grad():
    for entry in split["test"]:
        seq = load_clip(entry["path"])
        x = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        prob = torch.sigmoid(model(x)).item()
        all_probs.append(prob)
        all_labels.append(entry["label"])

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

print("Probability distribution overview:")
print(f"  Normal clips      -> min={all_probs[all_labels==0].min():.3f}  max={all_probs[all_labels==0].max():.3f}  mean={all_probs[all_labels==0].mean():.3f}")
print(f"  Shoplifting clips -> min={all_probs[all_labels==1].min():.3f}  max={all_probs[all_labels==1].max():.3f}  mean={all_probs[all_labels==1].mean():.3f}")

print("\n─── Threshold sweep ───")
print(f"{'Thresh':>7} {'Acc':>6} {'NormRecall':>11} {'ShopRecall':>11} {'NormFP':>7}")
for t in [0.50, 0.55, 0.60, 0.65, 0.68, 0.70, 0.72, 0.75, 0.80, 0.85, 0.90]:
    preds = (all_probs >= t).astype(int)
    acc = (preds == all_labels).mean()
    norm_recall = ((preds == 0) & (all_labels == 0)).sum() / (all_labels == 0).sum()
    shop_recall = ((preds == 1) & (all_labels == 1)).sum() / (all_labels == 1).sum()
    norm_fp = ((preds == 1) & (all_labels == 0)).sum()
    print(f"{t:7.2f} {acc:6.3f} {norm_recall:11.3f} {shop_recall:11.3f} {norm_fp:7d}")

Probability distribution overview:
  Normal clips      -> min=0.004  max=0.685  mean=0.255
  Shoplifting clips -> min=0.592  max=0.688  mean=0.672

─── Threshold sweep ───
 Thresh    Acc  NormRecall  ShopRecall  NormFP
   0.50  0.814       0.629       1.000      49
   0.55  0.818       0.636       1.000      48
   0.60  0.818       0.644       0.992      47
   0.65  0.807       0.652       0.962      46
   0.68  0.587       0.924       0.250      10
   0.70  0.500       1.000       0.000       0
   0.72  0.500       1.000       0.000       0
   0.75  0.500       1.000       0.000       0
   0.80  0.500       1.000       0.000       0
   0.85  0.500       1.000       0.000       0
   0.90  0.500       1.000       0.000       0


# v5 attention

In [ ]:
# ─── Cell 7: V5-style — BiLSTM + Temporal Attention ───
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

BASE_DIR   = "/content/drive/MyDrive/tame/dataset"
SPLIT_PATH = f"{BASE_DIR}/split.json"
SEQ_LEN    = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

with open(SPLIT_PATH) as f:
    split = json.load(f)

# ─── Dataset (position only, same as V0) ───
class ClipDataset(Dataset):
    def __init__(self, entries, seq_len=50):
        self.entries = entries
        self.seq_len = seq_len
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, idx):
        entry = self.entries[idx]
        seq = np.load(entry["path"]).astype(np.float32)
        if seq.shape[0] >= self.seq_len:
            seq = seq[-self.seq_len:]
        else:
            pad = np.zeros((self.seq_len - seq.shape[0], seq.shape[1]), dtype=np.float32)
            seq = np.vstack([seq, pad])
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(entry["label"], dtype=torch.float32)

train_ds = ClipDataset(split["train"], SEQ_LEN)
val_ds   = ClipDataset(split["val"], SEQ_LEN)
test_ds  = ClipDataset(split["test"], SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

# reuse V0 norm stats (same feature space, position-only)
X_mean = np.load(f"{BASE_DIR}/X_mean_v0.npy")
X_std  = np.load(f"{BASE_DIR}/X_std_v0.npy")
X_mean_t = torch.tensor(X_mean, dtype=torch.float32).to(device)
X_std_t  = torch.tensor(X_std, dtype=torch.float32).to(device)

# ─── Model: BiLSTM + additive temporal attention ───
class ShopliftingLSTMAttention(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        lstm_out_dim = hidden_size * 2

        # attention: score each timestep, softmax over time, weighted sum
        self.attn = nn.Sequential(
            nn.Linear(lstm_out_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x, return_attn=False):
        lstm_out, _ = self.lstm(x)                      # [B, T, 512]
        scores = self.attn(lstm_out).squeeze(-1)         # [B, T]
        weights = torch.softmax(scores, dim=1)            # [B, T] — which frames matter
        context = torch.sum(lstm_out * weights.unsqueeze(-1), dim=1)  # [B, 512] weighted sum

        out = self.classifier(context).squeeze()
        if return_attn:
            return out, weights
        return out

model = ShopliftingLSTMAttention().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

# ─── Training loop ───
EPOCHS = 50
PATIENCE = 8
best_val_loss = float("inf")
patience_counter = 0
best_model_path = f"{BASE_DIR}/vigiq_attn_best.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            x = (x - X_mean_t) / (X_std_t + 1e-8)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
    val_loss /= len(val_ds)

    print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest val_loss: {best_val_loss:.4f} — model saved to {best_model_path}")

# ─── Evaluation on TEST set ───
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(y.numpy().astype(int).tolist())

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average=None, labels=[0, 1])
cm = confusion_matrix(all_labels, all_preds)

print("\n─── ATTENTION MODEL — TEST SET RESULTS ───")
print(f"Accuracy: {acc:.4f}\n")
print("Class 0 = Normal, Class 1 = Shoplifting")
print(f"Normal      — precision={precision[0]:.3f}  recall={recall[0]:.3f}  f1={f1[0]:.3f}")
print(f"Shoplifting — precision={precision[1]:.3f}  recall={recall[1]:.3f}  f1={f1[1]:.3f}")
print(f"\nConfusion matrix:\n{cm}")
print(f"           (rows=actual, cols=predicted, order=[normal, shoplifting])")

all_probs = np.array(all_probs)
all_labels_arr = np.array(all_labels)
print(f"\nProbability spread:")
print(f"  Normal      -> min={all_probs[all_labels_arr==0].min():.3f} max={all_probs[all_labels_arr==0].max():.3f}")
print(f"  Shoplifting -> min={all_probs[all_labels_arr==1].min():.3f} max={all_probs[all_labels_arr==1].max():.3f}")

print("\n" + classification_report(all_labels, all_preds, target_names=["Normal", "Shoplifting"]))

Using device: cuda
Epoch  1 | train_loss=0.6740 | val_loss=0.6097
Epoch  2 | train_loss=0.5131 | val_loss=0.5235
Epoch  3 | train_loss=0.4756 | val_loss=0.5115
Epoch  4 | train_loss=0.4747 | val_loss=0.5080
Epoch  5 | train_loss=0.4669 | val_loss=0.5050
Epoch  6 | train_loss=0.4671 | val_loss=0.5229
Epoch  7 | train_loss=0.4624 | val_loss=0.5347
Epoch  8 | train_loss=0.4632 | val_loss=0.5039
Epoch  9 | train_loss=0.4577 | val_loss=0.5406
Epoch 10 | train_loss=0.4593 | val_loss=0.5209
Epoch 11 | train_loss=0.4569 | val_loss=0.5076
Epoch 12 | train_loss=0.4503 | val_loss=0.5224
Epoch 13 | train_loss=0.4527 | val_loss=0.5248
Epoch 14 | train_loss=0.4568 | val_loss=0.5318
Epoch 15 | train_loss=0.4509 | val_loss=0.5163
Epoch 16 | train_loss=0.4473 | val_loss=0.5088
Early stopping at epoch 16

Best val_loss: 0.5039 — model saved to /content/drive/MyDrive/tame/dataset/vigiq_attn_best.pth

─── ATTENTION MODEL — TEST SET RESULTS ───
Accuracy: 0.8068

Class 0 = Normal, Class 1 = Shoplifting
Norm

In [ ]:
import os
import numpy as np

BASE = '/content/drive/MyDrive/tame'
SRC_SHOPLIFTING = os.path.join(BASE, 'dataset', 'shoplifting')
OUT_WINDOWED = os.path.join(BASE, 'dataset', 'shoplifting_windowed')
os.makedirs(OUT_WINDOWED, exist_ok=True)

WINDOW_SIZE = 15
STRIDE = 5
KEEP_FROM_FRACTION = 0.5   # only keep windows starting from the back half of the clip

count = 0
for fname in os.listdir(SRC_SHOPLIFTING):
    if not fname.endswith('.npy'):
        continue
    arr = np.load(os.path.join(SRC_SHOPLIFTING, fname))  # shape (50, 34)
    total_frames = arr.shape[0]

    # drop trailing zero-padded frames first, so windows aren't built from padding
    real_len = total_frames
    while real_len > 0 and np.all(arr[real_len - 1] == 0):
        real_len -= 1
    if real_len < WINDOW_SIZE:
        continue  # too short to window meaningfully

    cutoff = int(real_len * KEEP_FROM_FRACTION)

    start = 0
    while start + WINDOW_SIZE <= real_len:
        if start >= cutoff:  # only keep windows from the back half
            window = arr[start:start + WINDOW_SIZE]
            out_name = f"{fname.replace('.npy','')}_w{start}.npy"
            np.save(os.path.join(OUT_WINDOWED, out_name), window)
            count += 1
        start += STRIDE

print(f"Saved {count} windowed shoplifting clips to {OUT_WINDOWED}")

Saved 2321 windowed shoplifting clips to /content/drive/MyDrive/tame/dataset/shoplifting_windowed


In [ ]:
import os
import numpy as np

BASE = '/content/drive/MyDrive/tame'
SRC_NORMAL = os.path.join(BASE, 'dataset', 'normal')
OUT_NORMAL_WINDOWED = os.path.join(BASE, 'dataset', 'normal_windowed')
os.makedirs(OUT_NORMAL_WINDOWED, exist_ok=True)

WINDOW_SIZE = 15
STRIDE = 5

count = 0
for fname in os.listdir(SRC_NORMAL):
    if not fname.endswith('.npy'):
        continue
    arr = np.load(os.path.join(SRC_NORMAL, fname))
    total_frames = arr.shape[0]

    real_len = total_frames
    while real_len > 0 and np.all(arr[real_len - 1] == 0):
        real_len -= 1
    if real_len < WINDOW_SIZE:
        continue

    start = 0
    while start + WINDOW_SIZE <= real_len:
        window = arr[start:start + WINDOW_SIZE]
        out_name = f"{fname.replace('.npy','')}_w{start}.npy"
        np.save(os.path.join(OUT_NORMAL_WINDOWED, out_name), window)
        count += 1
        start += STRIDE

print(f"Saved {count} windowed normal clips to {OUT_NORMAL_WINDOWED}")

Saved 6686 windowed normal clips to /content/drive/MyDrive/tame/dataset/normal_windowed


# windowed verison

In [ ]:
# ─── Cell 8: V3 — Train on "last-N-frames" windowed clips (event-near-end hypothesis) ───
import json
import numpy as np
import torch
import torch.nn as nn
import glob
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

random.seed(42)

BASE_DIR       = "/content/drive/MyDrive/tame/dataset"
NORMAL_DIR     = f"{BASE_DIR}/normal_windowed"
SHOPLIFT_DIR   = f"{BASE_DIR}/shoplifting_windowed"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

normal_files   = sorted(glob.glob(f"{NORMAL_DIR}/*.npy"))
shoplift_files = sorted(glob.glob(f"{SHOPLIFT_DIR}/*.npy"))

print(f"Normal windowed clips: {len(normal_files)}")
print(f"Shoplifting windowed clips: {len(shoplift_files)}")

# check actual shape/length of these new files
sample_normal = np.load(normal_files[0])
sample_shop   = np.load(shoplift_files[0])
print(f"Sample normal_windowed shape: {sample_normal.shape}")
print(f"Sample shoplifting_windowed shape: {sample_shop.shape}")

SEQ_LEN = sample_normal.shape[0]  # infer from actual data
assert sample_shop.shape[0] == SEQ_LEN, "Normal and shoplifting windows have different lengths — fix before training!"
print(f"Using SEQ_LEN = {SEQ_LEN}")

# ─── Fresh split on the NEW windowed data (separate from old split.json) ───
def split_files(files, train_r=0.70, val_r=0.15):
    files = files.copy()
    random.shuffle(files)
    n = len(files)
    n_train = int(n * train_r)
    n_val   = int(n * val_r)
    return files[:n_train], files[n_train:n_train+n_val], files[n_train+n_val:]

norm_train, norm_val, norm_test = split_files(normal_files)
shop_train, shop_val, shop_test = split_files(shoplift_files)

train_entries = [{"path": p, "label": 0} for p in norm_train] + [{"path": p, "label": 1} for p in shop_train]
val_entries   = [{"path": p, "label": 0} for p in norm_val]   + [{"path": p, "label": 1} for p in shop_val]
test_entries  = [{"path": p, "label": 0} for p in norm_test]  + [{"path": p, "label": 1} for p in shop_test]

random.shuffle(train_entries)
random.shuffle(val_entries)
random.shuffle(test_entries)

print(f"\nTrain: {len(train_entries)}  Val: {len(val_entries)}  Test: {len(test_entries)}")

# ─── Dataset ───
class WindowedDataset(Dataset):
    def __init__(self, entries, seq_len):
        self.entries = entries
        self.seq_len = seq_len
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, idx):
        entry = self.entries[idx]
        seq = np.load(entry["path"]).astype(np.float32)
        if seq.shape[0] != self.seq_len:
            # defensive pad/truncate in case of inconsistent lengths
            if seq.shape[0] > self.seq_len:
                seq = seq[-self.seq_len:]
            else:
                pad = np.zeros((self.seq_len - seq.shape[0], seq.shape[1]), dtype=np.float32)
                seq = np.vstack([seq, pad])
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(entry["label"], dtype=torch.float32)

train_ds = WindowedDataset(train_entries, SEQ_LEN)
val_ds   = WindowedDataset(val_entries, SEQ_LEN)
test_ds  = WindowedDataset(test_entries, SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

# ─── Normalization stats from TRAIN only ───
print("Computing normalization stats...")
all_train_seqs = np.vstack([np.load(e["path"]).astype(np.float32) for e in train_entries])
X_mean = all_train_seqs.mean(axis=0)
X_std  = all_train_seqs.std(axis=0)
np.save(f"{BASE_DIR}/X_mean_v3.npy", X_mean)
np.save(f"{BASE_DIR}/X_std_v3.npy", X_std)

X_mean_t = torch.tensor(X_mean, dtype=torch.float32).to(device)
X_std_t  = torch.tensor(X_std, dtype=torch.float32).to(device)

# ─── Model — same architecture as V0, input size = 34 (position only) ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :]).squeeze()

model = ShopliftingLSTM(input_size=sample_normal.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

# ─── Training loop ───
EPOCHS = 50
PATIENCE = 8
best_val_loss = float("inf")
patience_counter = 0
best_model_path = f"{BASE_DIR}/vigiq_v3_best.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            x = (x - X_mean_t) / (X_std_t + 1e-8)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
    val_loss /= len(val_ds)

    print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest val_loss: {best_val_loss:.4f} — model saved to {best_model_path}")

# ─── Evaluation on TEST set ───
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(y.numpy().astype(int).tolist())

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average=None, labels=[0, 1])
cm = confusion_matrix(all_labels, all_preds)

print("\n─── V3 LAST-N-FRAMES WINDOWED — TEST SET RESULTS ───")
print(f"Accuracy: {acc:.4f}\n")
print(f"Normal      — precision={precision[0]:.3f}  recall={recall[0]:.3f}  f1={f1[0]:.3f}")
print(f"Shoplifting — precision={precision[1]:.3f}  recall={recall[1]:.3f}  f1={f1[1]:.3f}")
print(f"\nConfusion matrix:\n{cm}")

all_probs_arr = np.array(all_probs)
all_labels_arr = np.array(all_labels)
print(f"\nProbability spread:")
print(f"  Normal      -> min={all_probs_arr[all_labels_arr==0].min():.3f} max={all_probs_arr[all_labels_arr==0].max():.3f}")
print(f"  Shoplifting -> min={all_probs_arr[all_labels_arr==1].min():.3f} max={all_probs_arr[all_labels_arr==1].max():.3f}")

print("\n" + classification_report(all_labels, all_preds, target_names=["Normal", "Shoplifting"]))

Using device: cuda
Normal windowed clips: 6686
Shoplifting windowed clips: 2321
Sample normal_windowed shape: (15, 34)
Sample shoplifting_windowed shape: (15, 34)
Using SEQ_LEN = 15

Train: 6304  Val: 1350  Test: 1353
Computing normalization stats...
Epoch  1 | train_loss=0.4790 | val_loss=0.4082
Epoch  2 | train_loss=0.3932 | val_loss=0.3966
Epoch  3 | train_loss=0.3841 | val_loss=0.3824
Epoch  4 | train_loss=0.3748 | val_loss=0.3877
Epoch  5 | train_loss=0.3692 | val_loss=0.3705
Epoch  6 | train_loss=0.3611 | val_loss=0.3632
Epoch  7 | train_loss=0.3504 | val_loss=0.3649
Epoch  8 | train_loss=0.3529 | val_loss=0.3569
Epoch  9 | train_loss=0.3468 | val_loss=0.3520
Epoch 10 | train_loss=0.3482 | val_loss=0.3496
Epoch 11 | train_loss=0.3436 | val_loss=0.3529
Epoch 12 | train_loss=0.3402 | val_loss=0.3416
Epoch 13 | train_loss=0.3339 | val_loss=0.3394
Epoch 14 | train_loss=0.3324 | val_loss=0.3401
Epoch 15 | train_loss=0.3288 | val_loss=0.3433
Epoch 16 | train_loss=0.3297 | val_loss=0.33

# v3 and weight imabalance fix

In [ ]:
# ─── Cell 9: V3 + class imbalance fix (pos_weight) ───
import json
import numpy as np
import torch
import torch.nn as nn
import glob
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

random.seed(42)

BASE_DIR       = "/content/drive/MyDrive/tame/dataset"
NORMAL_DIR     = f"{BASE_DIR}/normal_windowed"
SHOPLIFT_DIR   = f"{BASE_DIR}/shoplifting_windowed"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

normal_files   = sorted(glob.glob(f"{NORMAL_DIR}/*.npy"))
shoplift_files = sorted(glob.glob(f"{SHOPLIFT_DIR}/*.npy"))

sample_normal = np.load(normal_files[0])
SEQ_LEN = sample_normal.shape[0]
FEAT_DIM = sample_normal.shape[1]
print(f"SEQ_LEN={SEQ_LEN}, FEAT_DIM={FEAT_DIM}")

# ─── SAME split logic/seed as V3, so results are comparable ───
def split_files(files, train_r=0.70, val_r=0.15):
    files = files.copy()
    random.shuffle(files)
    n = len(files)
    n_train = int(n * train_r)
    n_val   = int(n * val_r)
    return files[:n_train], files[n_train:n_train+n_val], files[n_train+n_val:]

norm_train, norm_val, norm_test = split_files(normal_files)
shop_train, shop_val, shop_test = split_files(shoplift_files)

train_entries = [{"path": p, "label": 0} for p in norm_train] + [{"path": p, "label": 1} for p in shop_train]
val_entries   = [{"path": p, "label": 0} for p in norm_val]   + [{"path": p, "label": 1} for p in shop_val]
test_entries  = [{"path": p, "label": 0} for p in norm_test]  + [{"path": p, "label": 1} for p in shop_test]

random.shuffle(train_entries)
random.shuffle(val_entries)
random.shuffle(test_entries)

print(f"Train: {len(train_entries)}  Val: {len(val_entries)}  Test: {len(test_entries)}")

n_train_normal = sum(1 for e in train_entries if e["label"] == 0)
n_train_shop   = sum(1 for e in train_entries if e["label"] == 1)
pos_weight_value = n_train_normal / n_train_shop
print(f"Train class counts -> normal: {n_train_normal}, shoplifting: {n_train_shop}")
print(f"pos_weight = {pos_weight_value:.3f}  (tells the loss to treat each shoplifting example as this much more important)")

# ─── Dataset ───
class WindowedDataset(Dataset):
    def __init__(self, entries, seq_len):
        self.entries = entries
        self.seq_len = seq_len
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, idx):
        entry = self.entries[idx]
        seq = np.load(entry["path"]).astype(np.float32)
        if seq.shape[0] != self.seq_len:
            if seq.shape[0] > self.seq_len:
                seq = seq[-self.seq_len:]
            else:
                pad = np.zeros((self.seq_len - seq.shape[0], seq.shape[1]), dtype=np.float32)
                seq = np.vstack([seq, pad])
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(entry["label"], dtype=torch.float32)

train_ds = WindowedDataset(train_entries, SEQ_LEN)
val_ds   = WindowedDataset(val_entries, SEQ_LEN)
test_ds  = WindowedDataset(test_entries, SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

print("Computing normalization stats...")
all_train_seqs = np.vstack([np.load(e["path"]).astype(np.float32) for e in train_entries])
X_mean = all_train_seqs.mean(axis=0)
X_std  = all_train_seqs.std(axis=0)
np.save(f"{BASE_DIR}/X_mean_v3w.npy", X_mean)
np.save(f"{BASE_DIR}/X_std_v3w.npy", X_std)

X_mean_t = torch.tensor(X_mean, dtype=torch.float32).to(device)
X_std_t  = torch.tensor(X_std, dtype=torch.float32).to(device)

# ─── Model — identical architecture ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :]).squeeze()

model = ShopliftingLSTM(input_size=FEAT_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

# ─── ONLY CHANGE FROM V3: pos_weight on the loss ───
pos_weight_tensor = torch.tensor(pos_weight_value, dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

# ─── Training loop (identical to V3) ───
EPOCHS = 50
PATIENCE = 8
best_val_loss = float("inf")
patience_counter = 0
best_model_path = f"{BASE_DIR}/vigiq_v3w_best.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            x = (x - X_mean_t) / (X_std_t + 1e-8)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
    val_loss /= len(val_ds)

    print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest val_loss: {best_val_loss:.4f} — model saved to {best_model_path}")

# ─── Evaluation ───
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(y.numpy().astype(int).tolist())

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average=None, labels=[0, 1])
cm = confusion_matrix(all_labels, all_preds)

print("\n─── V3 + pos_weight — TEST SET RESULTS ───")
print(f"Accuracy: {acc:.4f}\n")
print(f"Normal      — precision={precision[0]:.3f}  recall={recall[0]:.3f}  f1={f1[0]:.3f}")
print(f"Shoplifting — precision={precision[1]:.3f}  recall={recall[1]:.3f}  f1={f1[1]:.3f}")
print(f"\nConfusion matrix:\n{cm}")

all_probs_arr = np.array(all_probs)
all_labels_arr = np.array(all_labels)
print(f"\nProbability spread:")
print(f"  Normal      -> min={all_probs_arr[all_labels_arr==0].min():.3f} max={all_probs_arr[all_labels_arr==0].max():.3f}")
print(f"  Shoplifting -> min={all_probs_arr[all_labels_arr==1].min():.3f} max={all_probs_arr[all_labels_arr==1].max():.3f}")

print("\n" + classification_report(all_labels, all_preds, target_names=["Normal", "Shoplifting"]))

Using device: cuda
SEQ_LEN=15, FEAT_DIM=34
Train: 6304  Val: 1350  Test: 1353
Train class counts -> normal: 4680, shoplifting: 1624
pos_weight = 2.882  (tells the loss to treat each shoplifting example as this much more important)
Computing normalization stats...
Epoch  1 | train_loss=0.7901 | val_loss=0.6988
Epoch  2 | train_loss=0.6810 | val_loss=0.6837
Epoch  3 | train_loss=0.6568 | val_loss=0.6370
Epoch  4 | train_loss=0.6212 | val_loss=0.6139
Epoch  5 | train_loss=0.6006 | val_loss=0.6112
Epoch  6 | train_loss=0.5915 | val_loss=0.6046
Epoch  7 | train_loss=0.5802 | val_loss=0.5988
Epoch  8 | train_loss=0.5773 | val_loss=0.5908
Epoch  9 | train_loss=0.5757 | val_loss=0.5855
Epoch 10 | train_loss=0.5658 | val_loss=0.5845
Epoch 11 | train_loss=0.5619 | val_loss=0.5765
Epoch 12 | train_loss=0.5608 | val_loss=0.5724
Epoch 13 | train_loss=0.5559 | val_loss=0.5617
Epoch 14 | train_loss=0.5530 | val_loss=0.5777
Epoch 15 | train_loss=0.5446 | val_loss=0.5551
Epoch 16 | train_loss=0.5363 | 

# sweep threshold test on plain v3

In [ ]:
# ─── Cell 10: Threshold sweep on V3 plain (no retraining) ───
import json
import numpy as np
import torch
import torch.nn as nn
import glob
import random

random.seed(42)

BASE_DIR       = "/content/drive/MyDrive/tame/dataset"
NORMAL_DIR     = f"{BASE_DIR}/normal_windowed"
SHOPLIFT_DIR   = f"{BASE_DIR}/shoplifting_windowed"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

normal_files   = sorted(glob.glob(f"{NORMAL_DIR}/*.npy"))
shoplift_files = sorted(glob.glob(f"{SHOPLIFT_DIR}/*.npy"))

sample_normal = np.load(normal_files[0])
SEQ_LEN = sample_normal.shape[0]
FEAT_DIM = sample_normal.shape[1]

# ─── SAME split logic/seed as V3, to get the exact same test set ───
def split_files(files, train_r=0.70, val_r=0.15):
    files = files.copy()
    random.shuffle(files)
    n = len(files)
    n_train = int(n * train_r)
    n_val   = int(n * val_r)
    return files[:n_train], files[n_train:n_train+n_val], files[n_train+n_val:]

norm_train, norm_val, norm_test = split_files(normal_files)
shop_train, shop_val, shop_test = split_files(shoplift_files)

test_entries = [{"path": p, "label": 0} for p in norm_test] + [{"path": p, "label": 1} for p in shop_test]
random.shuffle(test_entries)
print(f"Test set: {len(test_entries)} windows")

# ─── Load V3 plain model + its norm stats ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :]).squeeze()

model = ShopliftingLSTM(input_size=FEAT_DIM).to(device)
model.load_state_dict(torch.load(f"{BASE_DIR}/vigiq_v3_best.pth", map_location=device))
model.eval()

X_mean = np.load(f"{BASE_DIR}/X_mean_v3.npy")
X_std  = np.load(f"{BASE_DIR}/X_std_v3.npy")
X_mean_t = torch.tensor(X_mean, dtype=torch.float32).to(device)
X_std_t  = torch.tensor(X_std, dtype=torch.float32).to(device)

# ─── Get raw probabilities on test set ───
all_probs, all_labels = [], []
with torch.no_grad():
    for entry in test_entries:
        seq = np.load(entry["path"]).astype(np.float32)
        x = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        prob = torch.sigmoid(model(x)).item()
        all_probs.append(prob)
        all_labels.append(entry["label"])

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

print(f"\nProbability distribution:")
print(f"  Normal      -> min={all_probs[all_labels==0].min():.3f}  max={all_probs[all_labels==0].max():.3f}  mean={all_probs[all_labels==0].mean():.3f}")
print(f"  Shoplifting -> min={all_probs[all_labels==1].min():.3f}  max={all_probs[all_labels==1].max():.3f}  mean={all_probs[all_labels==1].mean():.3f}")

print("\n─── Threshold sweep ───")
print(f"{'Thresh':>7} {'Acc':>6} {'NormRecall':>11} {'ShopRecall':>11} {'ShopPrec':>9} {'NormFP':>7} {'ShopFN':>7}")
for t in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    preds = (all_probs >= t).astype(int)
    acc = (preds == all_labels).mean()
    norm_recall = ((preds == 0) & (all_labels == 0)).sum() / (all_labels == 0).sum()
    shop_recall = ((preds == 1) & (all_labels == 1)).sum() / (all_labels == 1).sum()
    shop_prec = ((preds == 1) & (all_labels == 1)).sum() / max((preds == 1).sum(), 1)
    norm_fp = ((preds == 1) & (all_labels == 0)).sum()
    shop_fn = ((preds == 0) & (all_labels == 1)).sum()
    print(f"{t:7.2f} {acc:6.3f} {norm_recall:11.3f} {shop_recall:11.3f} {shop_prec:9.3f} {norm_fp:7d} {shop_fn:7d}")

Test set: 1353 windows

Probability distribution:
  Normal      -> min=0.000  max=0.989  mean=0.140
  Shoplifting -> min=0.007  max=0.993  mean=0.641

─── Threshold sweep ───
 Thresh    Acc  NormRecall  ShopRecall  ShopPrec  NormFP  ShopFN
   0.20  0.793       0.748       0.923     0.560     253      27
   0.25  0.803       0.774       0.888     0.577     227      39
   0.30  0.819       0.808       0.851     0.606     193      52
   0.35  0.831       0.835       0.819     0.633     166      63
   0.40  0.836       0.856       0.779     0.652     145      77
   0.45  0.845       0.882       0.736     0.685     118      92
   0.50  0.844       0.898       0.688     0.702     102     109
   0.55  0.840       0.908       0.642     0.709      92     125
   0.60  0.841       0.927       0.593     0.739      73     142


# plain v3 with 1.5 pos weight

In [ ]:
# ─── Cell 9: V3 + class imbalance fix (pos_weight) ───
import json
import numpy as np
import torch
import torch.nn as nn
import glob
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

random.seed(42)

BASE_DIR       = "/content/drive/MyDrive/tame/dataset"
NORMAL_DIR     = f"{BASE_DIR}/normal_windowed"
SHOPLIFT_DIR   = f"{BASE_DIR}/shoplifting_windowed"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

normal_files   = sorted(glob.glob(f"{NORMAL_DIR}/*.npy"))
shoplift_files = sorted(glob.glob(f"{SHOPLIFT_DIR}/*.npy"))

sample_normal = np.load(normal_files[0])
SEQ_LEN = sample_normal.shape[0]
FEAT_DIM = sample_normal.shape[1]
print(f"SEQ_LEN={SEQ_LEN}, FEAT_DIM={FEAT_DIM}")

# ─── SAME split logic/seed as V3, so results are comparable ───
def split_files(files, train_r=0.70, val_r=0.15):
    files = files.copy()
    random.shuffle(files)
    n = len(files)
    n_train = int(n * train_r)
    n_val   = int(n * val_r)
    return files[:n_train], files[n_train:n_train+n_val], files[n_train+n_val:]

norm_train, norm_val, norm_test = split_files(normal_files)
shop_train, shop_val, shop_test = split_files(shoplift_files)

train_entries = [{"path": p, "label": 0} for p in norm_train] + [{"path": p, "label": 1} for p in shop_train]
val_entries   = [{"path": p, "label": 0} for p in norm_val]   + [{"path": p, "label": 1} for p in shop_val]
test_entries  = [{"path": p, "label": 0} for p in norm_test]  + [{"path": p, "label": 1} for p in shop_test]

random.shuffle(train_entries)
random.shuffle(val_entries)
random.shuffle(test_entries)

print(f"Train: {len(train_entries)}  Val: {len(val_entries)}  Test: {len(test_entries)}")

n_train_normal = sum(1 for e in train_entries if e["label"] == 0)
n_train_shop   = sum(1 for e in train_entries if e["label"] == 1)
pos_weight_value = n_train_normal / n_train_shop
print(f"Train class counts -> normal: {n_train_normal}, shoplifting: {n_train_shop}")
print(f"pos_weight = {pos_weight_value:.3f}  (tells the loss to treat each shoplifting example as this much more important)")

# ─── Dataset ───
class WindowedDataset(Dataset):
    def __init__(self, entries, seq_len):
        self.entries = entries
        self.seq_len = seq_len
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, idx):
        entry = self.entries[idx]
        seq = np.load(entry["path"]).astype(np.float32)
        if seq.shape[0] != self.seq_len:
            if seq.shape[0] > self.seq_len:
                seq = seq[-self.seq_len:]
            else:
                pad = np.zeros((self.seq_len - seq.shape[0], seq.shape[1]), dtype=np.float32)
                seq = np.vstack([seq, pad])
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(entry["label"], dtype=torch.float32)

train_ds = WindowedDataset(train_entries, SEQ_LEN)
val_ds   = WindowedDataset(val_entries, SEQ_LEN)
test_ds  = WindowedDataset(test_entries, SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

print("Computing normalization stats...")
all_train_seqs = np.vstack([np.load(e["path"]).astype(np.float32) for e in train_entries])
X_mean = all_train_seqs.mean(axis=0)
X_std  = all_train_seqs.std(axis=0)
np.save(f"{BASE_DIR}/X_mean_v3w.npy", X_mean)
np.save(f"{BASE_DIR}/X_std_v3w.npy", X_std)

X_mean_t = torch.tensor(X_mean, dtype=torch.float32).to(device)
X_std_t  = torch.tensor(X_std, dtype=torch.float32).to(device)

# ─── Model — identical architecture ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :]).squeeze()

model = ShopliftingLSTM(input_size=FEAT_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

# ─── ONLY CHANGE FROM V3: pos_weight on the loss ───
# Just change this one line in the cell above, everything else identical:
pos_weight_tensor = torch.tensor(1.5, dtype=torch.float32).to(device)  # was 2.9, now milder
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

# ─── Training loop (identical to V3) ───
EPOCHS = 50
PATIENCE = 8
best_val_loss = float("inf")
patience_counter = 0
best_model_path = f"{BASE_DIR}/vigiq_v3w_best.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            x = (x - X_mean_t) / (X_std_t + 1e-8)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
    val_loss /= len(val_ds)

    print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest val_loss: {best_val_loss:.4f} — model saved to {best_model_path}")

# ─── Evaluation ───
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(y.numpy().astype(int).tolist())

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average=None, labels=[0, 1])
cm = confusion_matrix(all_labels, all_preds)

print("\n─── V3 + pos_weight — TEST SET RESULTS ───")
print(f"Accuracy: {acc:.4f}\n")
print(f"Normal      — precision={precision[0]:.3f}  recall={recall[0]:.3f}  f1={f1[0]:.3f}")
print(f"Shoplifting — precision={precision[1]:.3f}  recall={recall[1]:.3f}  f1={f1[1]:.3f}")
print(f"\nConfusion matrix:\n{cm}")

all_probs_arr = np.array(all_probs)
all_labels_arr = np.array(all_labels)
print(f"\nProbability spread:")
print(f"  Normal      -> min={all_probs_arr[all_labels_arr==0].min():.3f} max={all_probs_arr[all_labels_arr==0].max():.3f}")
print(f"  Shoplifting -> min={all_probs_arr[all_labels_arr==1].min():.3f} max={all_probs_arr[all_labels_arr==1].max():.3f}")

print("\n" + classification_report(all_labels, all_preds, target_names=["Normal", "Shoplifting"]))

Using device: cuda
SEQ_LEN=15, FEAT_DIM=34
Train: 6304  Val: 1350  Test: 1353
Train class counts -> normal: 4680, shoplifting: 1624
pos_weight = 2.882  (tells the loss to treat each shoplifting example as this much more important)
Computing normalization stats...
Epoch  1 | train_loss=0.5863 | val_loss=0.5209
Epoch  2 | train_loss=0.4909 | val_loss=0.4874
Epoch  3 | train_loss=0.4757 | val_loss=0.4726
Epoch  4 | train_loss=0.4540 | val_loss=0.4707


# pos_weight mild + sweep threshold

In [3]:
# ─── Cell 11: V3 + mild pos_weight (1.5), then sweep threshold on it ───
import numpy as np
import torch
import torch.nn as nn
import glob
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix

random.seed(42)
BASE_DIR = "/content/drive/MyDrive/tame/dataset"
NORMAL_DIR = f"{BASE_DIR}/normal_windowed"
SHOPLIFT_DIR = f"{BASE_DIR}/shoplifting_windowed"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

normal_files = sorted(glob.glob(f"{NORMAL_DIR}/*.npy"))
shoplift_files = sorted(glob.glob(f"{SHOPLIFT_DIR}/*.npy"))
sample = np.load(normal_files[0])
SEQ_LEN, FEAT_DIM = sample.shape

def split_files(files, train_r=0.70, val_r=0.15):
    files = files.copy(); random.shuffle(files)
    n = len(files); n_train = int(n*train_r); n_val = int(n*val_r)
    return files[:n_train], files[n_train:n_train+n_val], files[n_train+n_val:]

norm_train, norm_val, norm_test = split_files(normal_files)
shop_train, shop_val, shop_test = split_files(shoplift_files)

train_entries = [{"path":p,"label":0} for p in norm_train] + [{"path":p,"label":1} for p in shop_train]
val_entries   = [{"path":p,"label":0} for p in norm_val]   + [{"path":p,"label":1} for p in shop_val]
test_entries  = [{"path":p,"label":0} for p in norm_test]  + [{"path":p,"label":1} for p in shop_test]
random.shuffle(train_entries); random.shuffle(val_entries); random.shuffle(test_entries)

class WindowedDataset(Dataset):
    def __init__(self, entries, seq_len):
        self.entries, self.seq_len = entries, seq_len
    def __len__(self): return len(self.entries)
    def __getitem__(self, idx):
        e = self.entries[idx]
        seq = np.load(e["path"]).astype(np.float32)
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(e["label"], dtype=torch.float32)

train_loader = DataLoader(WindowedDataset(train_entries, SEQ_LEN), batch_size=32, shuffle=True)
val_loader   = DataLoader(WindowedDataset(val_entries, SEQ_LEN), batch_size=32, shuffle=False)
test_loader  = DataLoader(WindowedDataset(test_entries, SEQ_LEN), batch_size=32, shuffle=False)

all_train_seqs = np.vstack([np.load(e["path"]).astype(np.float32) for e in train_entries])
X_mean, X_std = all_train_seqs.mean(axis=0), all_train_seqs.std(axis=0)
X_mean_t = torch.tensor(X_mean, dtype=torch.float32).to(device)
X_std_t  = torch.tensor(X_std, dtype=torch.float32).to(device)

class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size*2,128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64,1))
    def forward(self, x):
        out,_ = self.lstm(x)
        return self.classifier(out[:,-1,:]).squeeze()

model = ShopliftingLSTM(input_size=FEAT_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(1.5, dtype=torch.float32).to(device))

best_val_loss, patience_counter = float("inf"), 0
best_path = f"{BASE_DIR}/vigiq_v3w15_best.pth"

for epoch in range(1, 51):
    model.train(); train_loss = 0.0
    for x,y in train_loader:
        x,y = x.to(device), y.to(device)
        x = (x - X_mean_t)/(X_std_t+1e-8)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        train_loss += loss.item()*x.size(0)
    train_loss /= len(train_entries)

    model.eval(); val_loss = 0.0
    with torch.no_grad():
        for x,y in val_loader:
            x,y = x.to(device), y.to(device)
            x = (x - X_mean_t)/(X_std_t+1e-8)
            val_loss += criterion(model(x), y).item()*x.size(0)
    val_loss /= len(val_entries)
    print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss, patience_counter = val_loss, 0
        torch.save(model.state_dict(), best_path)
    else:
        patience_counter += 1
        if patience_counter >= 8:
            print(f"Early stopping at epoch {epoch}"); break

print(f"\nBest val_loss: {best_val_loss:.4f}")
model.load_state_dict(torch.load(best_path)); model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for x,y in test_loader:
        x = x.to(device)
        x = (x - X_mean_t)/(X_std_t+1e-8)
        probs = torch.sigmoid(model(x)).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(y.numpy().astype(int).tolist())

all_probs = np.array(all_probs); all_labels = np.array(all_labels)

print("\n─── Threshold sweep on pos_weight=1.5 model ───")
print(f"{'Thresh':>7} {'Acc':>6} {'NormRecall':>11} {'ShopRecall':>11} {'ShopPrec':>9}")
for t in [0.30, 0.35, 0.40, 0.50]:
    preds = (all_probs >= t).astype(int)
    acc = (preds == all_labels).mean()
    nr = ((preds==0)&(all_labels==0)).sum()/(all_labels==0).sum()
    sr = ((preds==1)&(all_labels==1)).sum()/(all_labels==1).sum()
    sp = ((preds==1)&(all_labels==1)).sum()/max((preds==1).sum(),1)
    print(f"{t:7.2f} {acc:6.3f} {nr:11.3f} {sr:11.3f} {sp:9.3f}")

Epoch  1 | train_loss=0.5781 | val_loss=0.5049
Epoch  2 | train_loss=0.4866 | val_loss=0.4869
Epoch  3 | train_loss=0.4662 | val_loss=0.4669
Epoch  4 | train_loss=0.4499 | val_loss=0.4652
Epoch  5 | train_loss=0.4421 | val_loss=0.4470
Epoch  6 | train_loss=0.4353 | val_loss=0.4540
Epoch  7 | train_loss=0.4324 | val_loss=0.4468
Epoch  8 | train_loss=0.4288 | val_loss=0.4360
Epoch  9 | train_loss=0.4249 | val_loss=0.4367
Epoch 10 | train_loss=0.4179 | val_loss=0.4501
Epoch 11 | train_loss=0.4179 | val_loss=0.4258
Epoch 12 | train_loss=0.4107 | val_loss=0.4166
Epoch 13 | train_loss=0.4092 | val_loss=0.4153
Epoch 14 | train_loss=0.4040 | val_loss=0.4112
Epoch 15 | train_loss=0.4052 | val_loss=0.4020
Epoch 16 | train_loss=0.3961 | val_loss=0.4179
Epoch 17 | train_loss=0.4015 | val_loss=0.4015
Epoch 18 | train_loss=0.3928 | val_loss=0.3957
Epoch 19 | train_loss=0.3919 | val_loss=0.4023
Epoch 20 | train_loss=0.3880 | val_loss=0.3940
Epoch 21 | train_loss=0.3781 | val_loss=0.3959
Epoch 22 | tr

# v3 with 0.35 threshold with temoporal attention

In [ ]:
# ─── Cell 12: V3 + Temporal Attention (on windowed last-15-frame data) ───
import numpy as np
import torch
import torch.nn as nn
import glob
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report

random.seed(42)
BASE_DIR     = "/content/drive/MyDrive/tame/dataset"
NORMAL_DIR   = f"{BASE_DIR}/normal_windowed"
SHOPLIFT_DIR = f"{BASE_DIR}/shoplifting_windowed"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

normal_files   = sorted(glob.glob(f"{NORMAL_DIR}/*.npy"))
shoplift_files = sorted(glob.glob(f"{SHOPLIFT_DIR}/*.npy"))
sample = np.load(normal_files[0])
SEQ_LEN, FEAT_DIM = sample.shape
print(f"SEQ_LEN={SEQ_LEN}, FEAT_DIM={FEAT_DIM}")

# ─── SAME split logic/seed as V3, so results are comparable ───
def split_files(files, train_r=0.70, val_r=0.15):
    files = files.copy(); random.shuffle(files)
    n = len(files); n_train = int(n*train_r); n_val = int(n*val_r)
    return files[:n_train], files[n_train:n_train+n_val], files[n_train+n_val:]

norm_train, norm_val, norm_test = split_files(normal_files)
shop_train, shop_val, shop_test = split_files(shoplift_files)

train_entries = [{"path":p,"label":0} for p in norm_train] + [{"path":p,"label":1} for p in shop_train]
val_entries   = [{"path":p,"label":0} for p in norm_val]   + [{"path":p,"label":1} for p in shop_val]
test_entries  = [{"path":p,"label":0} for p in norm_test]  + [{"path":p,"label":1} for p in shop_test]
random.shuffle(train_entries); random.shuffle(val_entries); random.shuffle(test_entries)
print(f"Train: {len(train_entries)}  Val: {len(val_entries)}  Test: {len(test_entries)}")

class WindowedDataset(Dataset):
    def __init__(self, entries, seq_len):
        self.entries, self.seq_len = entries, seq_len
    def __len__(self): return len(self.entries)
    def __getitem__(self, idx):
        e = self.entries[idx]
        seq = np.load(e["path"]).astype(np.float32)
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(e["label"], dtype=torch.float32)

train_loader = DataLoader(WindowedDataset(train_entries, SEQ_LEN), batch_size=32, shuffle=True)
val_loader   = DataLoader(WindowedDataset(val_entries, SEQ_LEN), batch_size=32, shuffle=False)
test_loader  = DataLoader(WindowedDataset(test_entries, SEQ_LEN), batch_size=32, shuffle=False)

all_train_seqs = np.vstack([np.load(e["path"]).astype(np.float32) for e in train_entries])
X_mean, X_std = all_train_seqs.mean(axis=0), all_train_seqs.std(axis=0)
np.save(f"{BASE_DIR}/X_mean_v3attn.npy", X_mean)
np.save(f"{BASE_DIR}/X_std_v3attn.npy", X_std)
X_mean_t = torch.tensor(X_mean, dtype=torch.float32).to(device)
X_std_t  = torch.tensor(X_std, dtype=torch.float32).to(device)

# ─── Model: BiLSTM + additive temporal attention ───
class ShopliftingLSTMAttention(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        lstm_out_dim = hidden_size * 2
        self.attn = nn.Sequential(
            nn.Linear(lstm_out_dim, 128), nn.Tanh(), nn.Linear(128, 1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x, return_attn=False):
        lstm_out, _ = self.lstm(x)                       # [B, T, 512]
        scores = self.attn(lstm_out).squeeze(-1)          # [B, T]
        weights = torch.softmax(scores, dim=1)             # [B, T]
        context = torch.sum(lstm_out * weights.unsqueeze(-1), dim=1)  # [B, 512]
        out = self.classifier(context).squeeze()
        if return_attn:
            return out, weights
        return out

model = ShopliftingLSTMAttention(input_size=FEAT_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

# ─── Training loop ───
best_val_loss, patience_counter = float("inf"), 0
best_path = f"{BASE_DIR}/vigiq_v3attn_best.pth"

for epoch in range(1, 51):
    model.train(); train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_entries)

    model.eval(); val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            x = (x - X_mean_t) / (X_std_t + 1e-8)
            val_loss += criterion(model(x), y).item() * x.size(0)
    val_loss /= len(val_entries)
    print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss, patience_counter = val_loss, 0
        torch.save(model.state_dict(), best_path)
    else:
        patience_counter += 1
        if patience_counter >= 8:
            print(f"Early stopping at epoch {epoch}"); break

print(f"\nBest val_loss: {best_val_loss:.4f} — saved to {best_path}")
model.load_state_dict(torch.load(best_path)); model.eval()

# ─── Evaluate at BOTH 0.5 and 0.35 for direct comparison ───Epoch 24 | train_loss=0.3007 | val_loss=0.3157
Epoch 25 | train_loss=0.3009 | val_loss=0.3103
all_probs, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        x = (x - X_mean_t) / (X_std_t + 1e-8)
        probs = torch.sigmoid(model(x)).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(y.numpy().astype(int).tolist())

all_probs = np.array(all_probs); all_labels = np.array(all_labels)

print(f"\nProbability spread:")
print(f"  Normal      -> min={all_probs[all_labels==0].min():.3f} max={all_probs[all_labels==0].max():.3f}")
print(f"  Shoplifting -> min={all_probs[all_labels==1].min():.3f} max={all_probs[all_labels==1].max():.3f}")

print("\n─── V3 + Attention — threshold comparison ───")
print(f"{'Thresh':>7} {'Acc':>6} {'NormRecall':>11} {'ShopRecall':>11} {'ShopPrec':>9}")
for t in [0.35, 0.50]:
    preds = (all_probs >= t).astype(int)
    acc = (preds == all_labels).mean()
    nr = ((preds==0)&(all_labels==0)).sum()/(all_labels==0).sum()
    sr = ((preds==1)&(all_labels==1)).sum()/(all_labels==1).sum()
    sp = ((preds==1)&(all_labels==1)).sum()/max((preds==1).sum(),1)
    print(f"{t:7.2f} {acc:6.3f} {nr:11.3f} {sr:11.3f} {sp:9.3f}")

    cm = confusion_matrix(all_labels, preds)
    print(f"  Confusion matrix:\n{cm}\n")

Using device: cuda
SEQ_LEN=15, FEAT_DIM=34
Train: 6304  Val: 1350  Test: 1353
Epoch  1 | train_loss=0.4667 | val_loss=0.4115
Epoch  2 | train_loss=0.3955 | val_loss=0.3943
Epoch  3 | train_loss=0.3796 | val_loss=0.3816
Epoch  4 | train_loss=0.3690 | val_loss=0.3837
Epoch  5 | train_loss=0.3616 | val_loss=0.3672
Epoch  6 | train_loss=0.3538 | val_loss=0.3618
Epoch  7 | train_loss=0.3513 | val_loss=0.3704
Epoch  8 | train_loss=0.3481 | val_loss=0.3842
Epoch  9 | train_loss=0.3469 | val_loss=0.3508
Epoch 10 | train_loss=0.3444 | val_loss=0.3456
Epoch 11 | train_loss=0.3428 | val_loss=0.3519
Epoch 12 | train_loss=0.3394 | val_loss=0.3444
Epoch 13 | train_loss=0.3334 | val_loss=0.3396
Epoch 14 | train_loss=0.3311 | val_loss=0.3318
Epoch 15 | train_loss=0.3276 | val_loss=0.3329
Epoch 16 | train_loss=0.3239 | val_loss=0.3380
Epoch 17 | train_loss=0.3253 | val_loss=0.3233
Epoch 18 | train_loss=0.3188 | val_loss=0.3215
Epoch 19 | train_loss=0.3166 | val_loss=0.3186
Epoch 20 | train_loss=0.3116 